In [ ]:
import gaze_heatmap_visualization as gv
import importlib
importlib.reload(gv)

#### Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import re
from datetime import datetime
import importlib

import gaze_tobiii_preprocessing as tgp
importlib.reload(tgp)  # Force reload the module
print("✅ Imported Tobii Gaze preprocessing module")


print("Libraries imported and reloaded successfully!")


#### Data Path Configuration

In [ ]:
# Parameter setup
from pathlib import Path

UMAP_DIR = Path.cwd().resolve()
if not (UMAP_DIR / "gaze_ml_prediction.py").exists() and (UMAP_DIR / "UMAP").exists():
    UMAP_DIR = UMAP_DIR / "UMAP"
base_path = UMAP_DIR.parent

# Directory paths
gaze_dir = base_path / "user_data/user_data_final/gaze"
response_dir = base_path / "user_data/user_data_final/responses"
cleaned_dir = base_path / "user_data/user_data_final/cleaned_gaze"
stimuli_dir = base_path / "user_data/user_data_final/stimuli"
metrics_csv_path = cleaned_dir / "eye_metrics_per_trial_all_participants_with_conditions.csv"
responses_csv_path = base_path / "user_data/user_data_final/responses.csv"
demographic_csv_path = base_path / "user_data/LLM-Reasoning-Lab-demographic.csv"
analysis_code_dir = base_path / "data_analysis_full"

participant_id = "3"  # Participant ID to analyze


#### ===========================================================================================
#### Printing AOI boundaries


In [ ]:
# tgp.print_participant_aoi_bounds(1, stimuli_dir)

### ===========================================================================================
### 📊 Gaze Point Visualization

Now let us visualize the transformed stimuli coordinate data and inspect the distribution of gaze points across different trials.

In [ ]:
# 🎯 Style comparison: academic vs. modern visualization
print("🎯 Showing a comparison of different visualization styles...")

participant_id = '0'
data_file_path = os.path.join(cleaned_dir, f"integrated_gaze_P-{participant_id}.csv")

if os.path.exists(data_file_path):
    # print(f"\n📊 Generating an academic-style heatmap (white background)...")
    # # Academic journal style - suitable for paper figures
    # data_academic = gv.quick_heatmap(
    #     participant_id=participant_id,
    #     data_file_path=data_file_path,
    #     max_trials=6,  # Show only the first 6 trials for easier comparison
    #     show_aoi=True,
    #     style='academic'  # Academic white-background style
    # )
    
    print(f"\n🎨 Generating a modern dark-style heatmap...")
    # Modern dark style - suitable for digital presentation and analysis
    data_modern = gv.quick_heatmap(
        participant_id=participant_id,
        data_file_path=data_file_path,
        max_trials=16,  # Show only the first 6 trials for easier comparison
        show_aoi=True,
        style='modern',  # Modern dark-background style
        use_original_aoi=False
    )

else:
    print(f"❌ Integrated data file not found: {data_file_path}")
    print("   Please run the earlier preprocessing steps first")

In [ ]:
# 🎯 Style comparison: academic vs. modern visualization
print("🎯 Showing a comparison of different visualization styles...")

participant_id = '8'
data_file_path = os.path.join(cleaned_dir, f"integrated_gaze_P-{participant_id}.csv")

if os.path.exists(data_file_path):
    print(f"\n📊 Generating an academic journal-style heatmap (white background)...")
    # Academic journal style - suitable for paper figures
    data_academic = gv.quick_heatmap(
        participant_id=participant_id,
        data_file_path=data_file_path,
        max_trials=16,  # Show only the first 6 trials for easier comparison
        show_aoi=True,
        style='academic',  # Academic white-background style
        use_original_aoi=False
    )
    
    # print(f"\n🎨 Generating a modern dark-style heatmap...")
    # # Modern dark style - suitable for digital presentation and analysis
    # data_modern = gv.quick_heatmap(
    #     participant_id=participant_id,
    #     data_file_path=data_file_path,
    #     max_trials=12,  # Show only the first 6 trials for easier comparison
    #     show_aoi=True,
    #     style='modern'  # Modern dark-background style
    # )

else:
    print(f"❌ Integrated data file not found: {data_file_path}")
    print("   Please run the earlier preprocessing steps first")

In [ ]:
# # 🎯 Saccade trajectory visualization
# print("🎯 Generating saccade trajectory visualization...")

# participant_id = '0'
# data_file_path = os.path.join(cleaned_dir, f"integrated_gaze_P-{participant_id}.csv")

# if os.path.exists(data_file_path):
#     print(f"\n📊 Generating the standard saccade trajectory plot...")
#     # Generate the saccade trajectory plot using the module
#     gv.quick_saccade_trajectory(
#         participant_id=participant_id,
#         data_file_path=data_file_path,
#         max_trials=6  # Display all trials
#     )
# else:
#     print(f"❌ Integrated data file not found: {data_file_path}")
#     print("   Please run the earlier preprocessing steps first")

In [ ]:
# print(f"\n✨ Generating the improved saccade trajectory plot with the new module...")
# # Test the improved saccade trajectory function
# gv.quick_saccade_improved(
#     participant_id=participant_id,
#     data_file_path=data_file_path,
#     max_trials=4  # Display all trials
# )

#### ===========================================================================================

#### integrate gaze data, Stimuli data, conditions

In [ ]:
# Use the integrate_participant function from the module 
integrated_data = tgp.integrate_participant(
    participant_id=participant_id,
    gaze_dir=gaze_dir,
    response_dir=response_dir,
    cleaned_dir=cleaned_dir,
    stimuli_dir=stimuli_dir,  # Added parameter for AOI assignment
    source='raw',
    save=True,
    save_cleaned=False,
    verbose=True,
    join_meta=True,
    force_reclean=False
)

In [ ]:
import pandas as pd
import numpy as np

def gaze_event_stats(csv_path):
    """
    Summarize basic information about gaze events.
    
    Parameters:
        csv_path: Path to the CSV file
    
    Returns:
        dict: Summary statistics
    """
    # Read data
    data = pd.read_csv(csv_path)
    
    # Basic information
    total_points = len(data)
    
    # Return an error if the `event_type` column is missing
    if 'event_type' not in data.columns:
        print("❌ The data does not contain an `event_type` column")
        return None
    
    # Count each event type
    fixation_count = (data['event_type'] == 'fixation').sum()
    saccade_count = (data['event_type'] == 'saccade').sum()
    
    # Compute proportions
    fixation_ratio = fixation_count / total_points * 100 if total_points > 0 else 0
    saccade_ratio = saccade_count / total_points * 100 if total_points > 0 else 0
    
    # Other event types (if any)
    other_count = total_points - fixation_count - saccade_count
    other_ratio = other_count / total_points * 100 if total_points > 0 else 0
    
    # Result dictionary
    results = {
        'total_points': total_points,
        'fixation_count': fixation_count,
        'fixation_ratio': fixation_ratio,
        'saccade_count': saccade_count,
        'saccade_ratio': saccade_ratio,
        'other_count': other_count,
        'other_ratio': other_ratio
    }
    
    # Print results
    print(f"\n📊 Gaze Event Statistics")
    print("="*40)
    print(f"Total data points: {total_points:,}")
    print(f"Fixation:     {fixation_count:,} ({fixation_ratio:.1f}%)")
    print(f"Saccade:      {saccade_count:,} ({saccade_ratio:.1f}%)")
    if other_count > 0:
        print(f"Other:         {other_count:,} ({other_ratio:.1f}%)")
    print("="*40)
    
    # If `trial_num` exists, summarize by trial
    if 'trial_num' in data.columns:
        print(f"\n📈 Summarize by trial")
        print("-"*40)
        for trial in sorted(data['trial_num'].unique()):
            trial_data = data[data['trial_num'] == trial]
            trial_total = len(trial_data)
            trial_fix = (trial_data['event_type'] == 'fixation').sum()
            trial_sac = (trial_data['event_type'] == 'saccade').sum()
            print(f"Trial {trial}: Total {trial_total:,} | "
                  f"Fix {trial_fix:,} ({trial_fix/trial_total*100:.1f}%) | "
                  f"Sac {trial_sac:,} ({trial_sac/trial_total*100:.1f}%)")
    
    return results


# Usage example
if __name__ == "__main__":
    # Example usage
    csv_path = cleaned_dir / "integrated_gaze_P-8.csv"
    
    # Get summary statistics
    stats = gaze_event_stats(csv_path)
    
    # Access specific values if needed
    if stats:
        print(f"\n💡 You can access specific values from the returned dictionary:")
        print(f"   stats['fixation_count'] = {stats['fixation_count']}")
        print(f"   stats['fixation_ratio'] = {stats['fixation_ratio']:.1f}%")

### =========================================================================================================================

### Batch-process gaze data for all participants

In [ ]:
# Specify participants to process (if `None`, all participants are detected automatically)
target_participants = None  # Or specify a list, e.g. ["0", "1", "2"]

# Step 1: Process individual participants (including AOI assignment)
process_summary = tgp.process_individual_participants(
    base_path=base_path,
    gaze_dir=gaze_dir,
    response_dir=response_dir, 
    output_dir=cleaned_dir,
    stimuli_dir=stimuli_dir,  # Added parameter for AOI assignment
    participant_list=target_participants,
    force_reprocess=False,
    verbose=True
)

print("\n" + "="*50)
print(f"Processing summary: successfully processed {process_summary['successful_participants']} participants")
print("="*50)

In [ ]:
# Step 2: Merge all participants
merged_data, merge_summary = tgp.merge_all_participants(
    base_path=base_path,
    output_dir="user_data/user_data_final/cleaned_gaze",  # Fixed: use the correct relative path
    participant_list=target_participants,
    output_filename="all_participants_integrated_gaze_with_aoi.csv",  # Updated filename to indicate AOI is included
    verbose=True
)

print("\n" + "="*50) 
print(f"Merge summary: {merge_summary['successful_files']} files merged successfully")
print(f"Total data points: {merge_summary['total_gaze_points']:,}")
print(f"Total number of trials: {merge_summary['total_trials']}")

# Check the AOI distribution
if merged_data is not None and len(merged_data) > 0 and 'aoi' in merged_data.columns:
    print(f"\n📍 Overall AOI distribution:")
    aoi_dist = merged_data['aoi'].value_counts()
    total_points = len(merged_data)
    for aoi, count in aoi_dist.items():
        pct = count / total_points * 100
        print(f"   {aoi}: {count:,} ({pct:.1f}%)")
else:
    print("⚠️ AOI information was not found in the merged data")
print("="*50)

## =================================================================================
## Posthoc pairwise comparisons


In [ ]:
# -*- coding: utf-8 -*-
"""
Compute and summarize AOI-level eye-tracking metrics (including TTFF / FFD / SFD).
Output 5 atomic AOIs + 2 merged AOIs:
  Atomic: claim, evidence, answer_aoi, reasoning_aoi, response_aoi
  Merged: claim_evidence (= claim ∪ evidence), answer_reasoning (= answer_aoi ∪ reasoning_aoi)
AOI spatial overlap is allowed: each AOI (including merged ones) is evaluated independently for hits.

Coordinate system: above the y-axis is +1, below is -1, therefore top >= bottom.

Metric description:
- <AOI>_gaze_duration            : cumulative duration of gaze episodes in this AOI (ms, summed after splitting by the gap threshold)
- <AOI>_fixation_duration        : cumulative duration of fixation episodes in this AOI (ms)
- <AOI>_fixation_count           : number of fixations (prefer counting via fixation_id; fall back to episode count if fixation_id is unavailable)
- <AOI>_saccade_length           : total Euclidean displacement of saccades (in stimuli-coordinate units)
- <AOI>_saccade_count            : number of saccade episodes (threshold-based) 
- <AOI>_pupil_diameter           : mean pupil diameter across all gaze samples hitting this AOI
- <AOI>_pupil_diameter_fixation  : mean pupil diameter within fixations
- <AOI>_ttff_ms                  : Time to First Fixation
- <AOI>_ffd_ms                   : First Fixation Duration
- <AOI>_sfd_ms                   : Single Fixation Duration

Update: supports individual claim/evidence and their union, and supports the answer_reasoning union; removed the forced early merge of claim/evidence.
Fix: if token_coordinates contains suffixed forms such as *"claim_aoi"* / *"evidence_aoi"*, the code now removes "_aoi" before recognition to avoid all-zero claim / evidence metrics.
"""

import os
import glob
import numpy as np
import pandas as pd
import re


# Atomic and merged AOI definitions
ATOM_AOIS = ['claim', 'evidence', 'answer_aoi', 'reasoning_aoi', 'response_aoi']
COMBINED_AOIS = [
    ('claim_evidence', ['claim', 'evidence']),
    ('answer_reasoning', ['answer_aoi', 'reasoning_aoi']),
]
ALL_AOIS = ATOM_AOIS + [c[0] for c in COMBINED_AOIS]

FIX_EPISODE_GAP_US = 100_000    # 100ms fixation episode threshold for splitting episodes
SACCADE_EPISODE_GAP_US = 50_000  # 50ms saccade episode threshold for splitting episodes
CLAMP_RANGE = (-1.0, 1.0)
SAVE_INTERMEDIATE = False  # Keep this switch if an intermediate non-backfilled version is needed

print("🔍 Computing eye-tracking metrics: atomic + merged AOIs, with overlap allowed ...")

# ----------------- AOI boundary handling ----------------- #

def _expand_reasoning(top: float, bottom: float) -> tuple:
    return top - 0.00, bottom - 0.00

def _expand_answer(top: float, bottom: float) -> tuple:
    return top + 0.00, bottom - 0.00

def _expand_response(top: float, bottom: float) -> tuple:
    return top + 0.00, bottom

def _clamp_rect(left, right, top, bottom, lo=-1.0, hi=1.0):
    left   = float(np.clip(left,   lo, hi))
    right  = float(np.clip(right,  lo, hi))
    top    = float(np.clip(top,    lo, hi))
    bottom = float(np.clip(bottom, lo, hi))
    return left, right, top, bottom

def _ensure_top_bottom(top, bottom):
    if top < bottom:
        top, bottom = bottom, top
    return top, bottom

def _union_rects(rects):
    if not rects:
        return None
    left   = min(r[0] for r in rects)
    right  = max(r[1] for r in rects)
    top    = max(r[2] for r in rects)
    bottom = min(r[3] for r in rects)
    left, right, top, bottom = _clamp_rect(left, right, top, bottom, *CLAMP_RANGE)
    top, bottom = _ensure_top_bottom(top, bottom)
    return (left, right, top, bottom)

def _read_and_prepare_bounds_for_pid(pid: str, stimuli_dir: str):
    fpath = os.path.join(stimuli_dir, f"token_coordinates_P-{pid}.csv")
    if not os.path.exists(fpath):
        print(f"⚠️ token_coordinates not found: {fpath}; falling back to label-hit mode")
        return None
    df = pd.read_csv(fpath)
    need = ['trial_num', 'stimulus_id', 'section', 'aoi_left', 'aoi_right', 'aoi_top', 'aoi_bottom']
    miss = [c for c in need if c not in df.columns]
    if miss:
        print(f"⚠️ token_coordinates is missing columns {miss}; falling back to label-hit mode")
        return None
    tdf = df.dropna(subset=['aoi_left','aoi_right','aoi_top','aoi_bottom']).copy()
    if tdf.empty:
        print("⚠️ token_coordinates has no valid boundaries; falling back to label-hit mode")
        return None

    # Key step: remove the `_aoi` suffix before standardization to ensure correct claim/evidence recognition
    tdf['section_norm'] = (tdf['section'].astype(str)
                           .str.strip().str.lower()
                           .str.replace('_aoi','', regex=False))

    for c in ['aoi_left','aoi_right','aoi_top','aoi_bottom']:
        tdf[c] = pd.to_numeric(tdf[c], errors='coerce')

    out = {}
    base_names = {'claim','evidence','answer','reasoning','response'}
    for trial, g in tdf.groupby('trial_num'):
        rows = []
        for _, r in g.iterrows():
            name = r['section_norm']
            if name not in base_names:
                continue
            # Map to the final atomic AOI name
            if name in ['answer','reasoning','response']:
                aoiname = f"{name}_aoi"
            else:
                aoiname = name  # Keep claim/evidence as-is
            left   = float(r['aoi_left'])
            right  = float(r['aoi_right'])
            top    = float(max(r['aoi_top'], r['aoi_bottom']))
            bottom = float(min(r['aoi_top'], r['aoi_bottom']))
            if aoiname == 'answer_aoi':
                top, bottom = _expand_answer(top, bottom)
            elif aoiname == 'reasoning_aoi':
                top, bottom = _expand_reasoning(top, bottom)
            elif aoiname == 'response_aoi':
                top, bottom = _expand_response(top, bottom)
            top, bottom = _ensure_top_bottom(top, bottom)
            left, right, top, bottom = _clamp_rect(left, right, top, bottom, *CLAMP_RANGE)
            rows.append({'aoi_name': aoiname, 'left': left, 'right': right, 'top': top, 'bottom': bottom})

        trial_bounds = {r['aoi_name']: (r['left'], r['right'], r['top'], r['bottom']) for r in rows}
        # Build merged AOIs
        ce_union = _union_rects([trial_bounds.get(x) for x in ['claim','evidence'] if trial_bounds.get(x)])
        if ce_union is not None:
            trial_bounds['claim_evidence'] = ce_union
        ar_union = _union_rects([trial_bounds.get(x) for x in ['answer_aoi','reasoning_aoi'] if trial_bounds.get(x)])
        if ar_union is not None:
            trial_bounds['answer_reasoning'] = ar_union
        out[trial] = trial_bounds

    # Debug output (first-trial example only)
    if out:
        first_trial = sorted(out.keys())[0]
        tb = out[first_trial]
        present = [k for k in ['claim','evidence','answer_aoi','reasoning_aoi','response_aoi','claim_evidence','answer_reasoning'] if k in tb]
        print(f"🧪 Debug: PID {pid} first trial ({first_trial}) contains AOIs: {present}")
    return out

def _point_in_rect(df: pd.DataFrame, rect):
    l, r, t, b = rect
    return (df['stimuli_x'].astype(float).between(l, r, inclusive='both') &
            df['stimuli_y'].astype(float).between(b, t, inclusive='both'))

# ----------------- Metric calculation ----------------- #

def _split_fixation_episodes(fix_df: pd.DataFrame, gap_us: int = FIX_EPISODE_GAP_US):
    if fix_df is None or len(fix_df) == 0:
        return [], 0
    f = fix_df.sort_values('device_time_stamp').reset_index(drop=True).copy()
    gaps = f['device_time_stamp'].diff().fillna(pd.NA)
    new_mask = gaps.isna() | (gaps > gap_us)
    grp = new_mask.cumsum()
    episodes = []
    for _, gdf in f.groupby(grp):
        start_us = int(gdf['device_time_stamp'].iloc[0])
        end_us   = int(gdf['device_time_stamp'].iloc[-1])
        dur_ms   = (end_us - start_us) / 1000.0
        dur_ms   = dur_ms if dur_ms >= 0 else np.nan
        episodes.append({'start_us': start_us, 'end_us': end_us, 'dur_ms': dur_ms})
    return episodes, len(episodes)

def calculate_eye_metrics_per_trial(participant_data: pd.DataFrame, participant_id: str, stimuli_dir: str):
    bounds_all = _read_and_prepare_bounds_for_pid(participant_id, stimuli_dir)

    pdata = participant_data.copy()
    pdata['aoi_std'] = (pdata['aoi'].astype(str).str.strip().str.lower())
    # Map answer/reasoning/response consistently
    map_extra = {'answer': 'answer_aoi', 'reasoning': 'reasoning_aoi', 'response': 'response_aoi'}
    pdata['aoi_std'] = pdata['aoi_std'].replace(map_extra)

    pdata['stimuli_x'] = pd.to_numeric(pdata.get('stimuli_x'), errors='coerce')
    pdata['stimuli_y'] = pd.to_numeric(pdata.get('stimuli_y'), errors='coerce')

    if pdata['trial_num'].notna().sum() == 0:
        print(f"   ⚠️ Participant {participant_id}: no valid trial_num")
        return None

    def _episode_sum_ms(df: pd.DataFrame, gap_us: int) -> float:
        """Sum of episode durations after splitting by time gaps (us)."""
        if df is None or len(df) <= 1:
            return 0.0
        ts = pd.to_numeric(df.get('device_time_stamp'), errors='coerce').dropna()
        if len(ts) <= 1:
            return 0.0
        ts = ts.sort_values().reset_index(drop=True)
        gaps = ts.diff().fillna(pd.NA)
        new_mask = gaps.isna() | (gaps > gap_us)
        grp = new_mask.cumsum()
        total_ms = 0.0
        for _, g in ts.groupby(grp):
            start_us = float(g.iloc[0])
            end_us = float(g.iloc[-1])
            dur_ms = (end_us - start_us) / 1000.0
            if dur_ms >= 0:
                total_ms += dur_ms
        return float(total_ms)

    trial_metrics = []
    for trial_num in sorted(pdata['trial_num'].dropna().unique()):
        tdf = pdata[pdata['trial_num'] == trial_num].copy()
        tdf = tdf.sort_values('device_time_stamp').reset_index(drop=True)
        if tdf.empty:
            continue
        trial_start_us = int(tdf['device_time_stamp'].min())
        trial_bounds = bounds_all.get(trial_num, {}) if isinstance(bounds_all, dict) else {}
        row_out = {'participant_id': participant_id, 'trial_num': int(trial_num)}

        for aoi in ALL_AOIS:
            if trial_bounds:  # Geometric boundaries are available
                rect = trial_bounds.get(aoi)
                if rect is None:
                    if aoi == 'claim_evidence':
                        rect = _union_rects([trial_bounds.get(x) for x in ['claim','evidence'] if trial_bounds.get(x)])
                    elif aoi == 'answer_reasoning':
                        rect = _union_rects([trial_bounds.get(x) for x in ['answer_aoi','reasoning_aoi'] if trial_bounds.get(x)])
                if rect is None:
                    aoi_df = tdf.iloc[0:0]
                else:
                    aoi_df = tdf[_point_in_rect(tdf, rect)].copy()
            else:  # No boundaries: use label-based logic
                if aoi in ATOM_AOIS:
                    aoi_df = tdf[tdf['aoi_std'] == aoi].copy()
                elif aoi == 'claim_evidence':
                    aoi_df = tdf[tdf['aoi_std'].isin(['claim','evidence'])].copy()
                elif aoi == 'answer_reasoning':
                    aoi_df = tdf[tdf['aoi_std'].isin(['answer_aoi','reasoning_aoi'])].copy()
                else:
                    aoi_df = tdf.iloc[0:0]

            if aoi_df.empty:
                row_out.update({
                    f'{aoi}_gaze_duration': 0,
                    f'{aoi}_fixation_duration': 0,
                    f'{aoi}_fixation_count': 0,
                    f'{aoi}_saccade_length': 0,
                    f'{aoi}_saccade_count': 0,
                    f'{aoi}_pupil_diameter': np.nan,
                    f'{aoi}_pupil_diameter_fixation': np.nan,
                    f'{aoi}_ttff_ms': np.nan,
                    f'{aoi}_ffd_ms': np.nan,
                    f'{aoi}_sfd_ms': np.nan,
                })
                continue

            fix = aoi_df[aoi_df['event_type'] == 'fixation'].copy()
            sac = aoi_df[aoi_df['event_type'] == 'saccade'].copy()

            def _saccade_distance(df):
                if df is None or len(df) <= 1:
                    return 0
                dx = df['stimuli_x'].diff().dropna()
                dy = df['stimuli_y'].diff().dropna()
                return np.sqrt(dx**2 + dy**2).sum()

            # gaze_duration: accumulate episode durations within AOI (avoid counting large gaps while out of AOI)
            gaze_duration = _episode_sum_ms(aoi_df, gap_us=FIX_EPISODE_GAP_US)

            if len(fix) > 0:
                fix_episodes, n_fix_episodes = _split_fixation_episodes(fix, gap_us=FIX_EPISODE_GAP_US)
                fixation_duration = float(np.nansum([e.get('dur_ms', np.nan) for e in fix_episodes]))
                if 'fixation_id' in fix.columns and fix['fixation_id'].notna().any():
                    fixation_count = int(pd.to_numeric(fix['fixation_id'], errors='coerce').nunique(dropna=True))
                else:
                    fixation_count = int(n_fix_episodes)
            else:
                fix_episodes, n_fix_episodes = [], 0
                fixation_duration = 0.0
                fixation_count = 0

            saccade_length = _saccade_distance(sac)
            if len(sac) > 0:
                s_sorted = sac.sort_values('device_time_stamp')
                gaps = s_sorted['device_time_stamp'].diff()
                saccade_count = ((gaps > SACCADE_EPISODE_GAP_US) | gaps.isna()).sum()
            else:
                saccade_count = 0

            pupil_diameter = (aoi_df['pupil_diameter'].mean()
                               if 'pupil_diameter' in aoi_df.columns and aoi_df['pupil_diameter'].notna().sum() > 0
                               else np.nan)
            pupil_diameter_fixation = (fix['pupil_diameter'].mean()
                                        if len(fix) > 0 and 'pupil_diameter' in fix.columns and fix['pupil_diameter'].notna().sum() > 0
                                        else np.nan)

            if len(fix) > 0:
                first_fix_us = int(fix['device_time_stamp'].min())
                ttff_ms = (first_fix_us - trial_start_us) / 1000.0
                ttff_ms = ttff_ms if ttff_ms >= 0 else np.nan
                episodes = fix_episodes
                n_epi = n_fix_episodes
                if n_epi > 0:
                    first_ep = min(episodes, key=lambda d: d['start_us'])
                    ffd_ms = first_ep['dur_ms']
                    sfd_ms = first_ep['dur_ms'] if n_epi == 1 else np.nan
                else:
                    ffd_ms = np.nan
                    sfd_ms = np.nan
            else:
                ttff_ms = np.nan
                ffd_ms = np.nan
                sfd_ms = np.nan

            row_out.update({
                f'{aoi}_gaze_duration': gaze_duration,
                f'{aoi}_fixation_duration': fixation_duration,
                f'{aoi}_fixation_count': fixation_count,
                f'{aoi}_saccade_length': saccade_length,
                f'{aoi}_saccade_count': saccade_count,
                f'{aoi}_pupil_diameter': pupil_diameter,
                f'{aoi}_pupil_diameter_fixation': pupil_diameter_fixation,
                f'{aoi}_ttff_ms': ttff_ms,
                f'{aoi}_ffd_ms': ffd_ms,
                f'{aoi}_sfd_ms': sfd_ms,
            })

        trial_metrics.append(row_out)

    return pd.DataFrame(trial_metrics) if trial_metrics else None


# ---------- Paths ----------
cleaned_dir_full = os.path.join(base_path, "user_data/user_data_final/cleaned_gaze")
responses_dir    = os.path.join(base_path, "user_data/user_data_final/responses")
stimuli_dir      = os.path.join(base_path, "user_data/user_data_final/stimuli")

print("📂 Scanning participant files ...")
participant_files = [f for f in os.listdir(cleaned_dir_full)
                     if f.startswith("integrated_gaze_P-") and f.endswith(".csv")]
print(f"✅ Found {len(participant_files)} files")

all_trial_metrics = []
successful_participants = 0
for file in sorted(participant_files):
    participant_id = file.split("integrated_gaze_P-")[1].split(".csv")[0]
    fpath = os.path.join(cleaned_dir_full, file)
    try:
        print(f"  ▶ Processing participant {participant_id} ...")
        dfp = pd.read_csv(fpath)
        tm = calculate_eye_metrics_per_trial(dfp, participant_id, stimuli_dir)
        if tm is not None:
            all_trial_metrics.append(tm)
            successful_participants += 1
            print(f"    ✅ {participant_id}: {len(tm)} trials")
        else:
            print(f"    ❌ {participant_id}: no trial metrics")
    except Exception as e:
        print(f"    ❌ {participant_id}: error {e}")

print(f"📊 Success: {successful_participants}/{len(participant_files)}")

if all_trial_metrics:
    final_metrics = pd.concat(all_trial_metrics, ignore_index=True)
    print(f"✅ Merged trial metrics: {len(final_metrics)} rows")
    if SAVE_INTERMEDIATE:
        mid_path = os.path.join(cleaned_dir_full, "eye_metrics_per_trial_all_participants.csv")
        final_metrics.to_csv(mid_path, index=False)
        print(f"💾 Intermediate file: {mid_path}")
    else:
        print("⏭️ Skipping intermediate-version save")

    print("🔗 Backfilling condition / trial_type / reasoning_type ...")

    def _find_trial_col(df: pd.DataFrame) -> str:
        for c in ["trial_num", "trial", "trial_id", "trialIndex", "trial_index", "trialNo", "trial_idx"]:
            if c in df.columns:
                return c
        df["trial_num"] = pd.NA
        return "trial_num"

    def _norm_pid(v):
        if pd.isna(v): return ""
        s = str(v).strip()
        if s.isdigit(): return s
        m = re.search(r"(\d+)$", s)
        return str(int(m.group(1))) if m else s

    def _to_int_or_none(v):
        if pd.isna(v): return None
        s = str(v).strip()
        m = re.search(r"(-?\d+)$", s)
        if not m:
            try:
                return int(float(s))
            except Exception:
                return None
        try:
            return int(m.group(1))
        except Exception:
            return None

    final_metrics['participant_id'] = final_metrics['participant_id'].map(_norm_pid)
    final_metrics['trial_num'] = final_metrics['trial_num'].map(_to_int_or_none)
    for c in ['condition','trial_type','reasoning_type']:
        if c not in final_metrics.columns:
            final_metrics[c] = pd.NA

    responses_cache = {}
    for pid, sub_idx in final_metrics.groupby('participant_id').groups.items():
        resp_path = os.path.join(responses_dir, f"all_responses_P-{pid}.csv")
        if pid not in responses_cache:
            if os.path.exists(resp_path):
                try:
                    rdf = pd.read_csv(resp_path)
                    tcol = _find_trial_col(rdf)
                    if tcol != 'trial_num':
                        rdf = rdf.rename(columns={tcol:'trial_num'})
                    rdf['participant_id'] = rdf.get('participant_id', pid)
                    rdf['participant_id'] = rdf['participant_id'].map(_norm_pid)
                    rdf['trial_num'] = rdf['trial_num'].map(_to_int_or_none)
                    for c in ['condition','trial_type','reasoning_type']:
                        if c not in rdf.columns:
                            rdf[c] = pd.NA
                    responses_cache[pid] = rdf[['participant_id','trial_num','condition','trial_type','reasoning_type']]
                except Exception as e:
                    print(f"❌ Failed to read {resp_path}: {e}")
                    responses_cache[pid] = pd.DataFrame(columns=['participant_id','trial_num','condition','trial_type','reasoning_type'])
            else:
                print(f"❌ Missing responses file: {resp_path}")
                responses_cache[pid] = pd.DataFrame(columns=['participant_id','trial_num','condition','trial_type','reasoning_type'])
        rdf = responses_cache[pid]
        for idx in sub_idx:
            t = final_metrics.at[idx, 'trial_num']
            if t is None:
                print(f"[MISS] pid={pid} trial=None")
                continue
            hit = rdf[(rdf['participant_id']==pid) & (rdf['trial_num']==t)]
            if hit.empty:
                print(f"[MISS] pid={pid} trial={t}")
                continue
            row = hit.iloc[-1]
            final_metrics.at[idx,'condition'] = row['condition']
            final_metrics.at[idx,'trial_type'] = row['trial_type']
            final_metrics.at[idx,'reasoning_type'] = row['reasoning_type']
            print(f"[OK] pid={pid} trial={t} -> cond={row['condition']} trial_type={row['trial_type']} reasoning_type={row['reasoning_type']}")

    def classify_condition(row):
        # if row['condition'] == 'without_reasoning' and str(row['reasoning_type']).strip().lower() in ('none','nan',''):
        #     return 'no_reasoning'
        if row['condition'] == 'without_reasoning' and row['trial_type'] == 'pre':
            return 'no_answer'
        elif row['condition'] == 'without_reasoning' and row['trial_type'] == 'main':
            return 'no_reasoning'
        elif row['condition'] == 'with_reasoning' and row['reasoning_type'] == 'correct':
            return 'correct_reasoning'
        elif row['condition'] == 'with_reasoning' and pd.notna(row['reasoning_type']) and row['reasoning_type'] != 'correct':
            return 'incorrect_reasoning'
        else:
            return 'other'

    final_metrics['classified_condition'] = final_metrics.apply(classify_condition, axis=1)

    out_path = os.path.join(cleaned_dir_full, 'eye_metrics_per_trial_all_participants_with_conditions.csv')
    final_metrics.to_csv(out_path, index=False)
    print(f"\n💾 Saved final file (including atomic + merged AOI metrics): {out_path}")
else:
    print("❌ No gaze metrics available to merge")

# Note: AOIs are no longer mutually exclusive; merged AOIs are independent union metrics and do not affect atomic AOI statistics.


# ====================================================================================
# Gaze statistics analysis

In [ ]:
import gaze_plot_pairwise_ttest as gzplot

In [ ]:
"""Participant-level AOI × condition main-effect test (per metric)"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2
from typing import List, Optional


def analyze_aoi_condition_effects(
    df: pd.DataFrame,
    metrics: List[str],
    alpha: float = 0.05,
    use_log10: bool = False
) -> pd.DataFrame:
    """
    Test main effect of classified_condition on gaze metrics for each AOI.
    
    For each gaze metric and AOI column (pattern: f"{AOI}_{metric}"),
    tests whether there is a main effect of classified_condition at participant level.
    
    Parameters
    ----------
    df : pd.DataFrame
        Data with columns: participant_id, classified_condition, {AOI}_{metric}
    metrics : List[str]
        List of metric names to analyze (e.g., ['fixation_count', 'dwell_time'])
    alpha : float, default=0.05
        Significance level for hypothesis testing
    use_log10 : bool, default=False
        Whether to log10-transform values before analysis
    
    Returns
    -------
    pd.DataFrame
        Results with columns: metric, AOI, test, statistic, df, p_value, 
        significant, n_participants, n_observations, note
    
    Notes
    -----
    Primary method: MixedLM LRT comparing 
        value ~ C(classified_condition) vs value ~ 1
        with random intercept for participant_id
    
    Fallback: OLS with participant fixed effects + cluster-robust SE 
        and joint F-test on condition dummies
    """
    
    def _detect_aois_for_metric(df: pd.DataFrame, metric: str) -> List[str]:
        """Extract AOI names from column names ending with _{metric}"""
        suffix = f"_{metric}"
        aois = {col[:-len(suffix)] for col in df.columns if col.endswith(suffix)}
        return sorted(aois)
    
    def _create_empty_result(metric: str, aoi: Optional[str], note: str, 
                            n_participants: int = 0, n_observations: int = 0) -> dict:
        """Create a result row for failed analyses"""
        return {
            'metric': metric,
            'AOI': aoi,
            'test': 'NA',
            'statistic': np.nan,
            'df': np.nan,
            'p_value': np.nan,
            'significant': False,
            'n_participants': n_participants,
            'n_observations': n_observations,
            'note': note
        }
    
    def _prepare_data(df: pd.DataFrame, aoi: str, metric: str, 
                     use_log10: bool) -> Optional[pd.DataFrame]:
        """Extract and aggregate data for a specific AOI-metric combination"""
        col = f"{aoi}_{metric}"
        if col not in df.columns:
            return None
        
        # Extract relevant columns
        sub = df[['participant_id', 'classified_condition', col]].copy()
        sub = sub.rename(columns={col: 'value'})
        sub = sub.dropna(subset=['participant_id', 'classified_condition', 'value'])
        
        if sub.empty:
            return None
        
        # Aggregate to participant × condition means
        agg = sub.groupby(
            ['participant_id', 'classified_condition'], 
            as_index=False
        )['value'].mean()
        
        # Apply log10 transformation if requested
        if use_log10:
            eps = 1e-6
            vmin = agg['value'].min()
            shift = max(0.0, -(vmin - eps))
            agg['value'] = np.log10(agg['value'] + shift + eps)
        
        agg['participant_id'] = agg['participant_id'].astype(str)
        return agg
    
    def _run_mixedlm_lrt(agg: pd.DataFrame, k: int, alpha: float) -> Optional[dict]:
        """Run Mixed Linear Model Likelihood Ratio Test"""
        try:
            # Null model: intercept only
            m0 = smf.mixedlm('value ~ 1', data=agg, groups=agg['participant_id'])
            r0 = m0.fit(method='lbfgs', reml=False, maxiter=200, disp=False)
            
            # Alternative model: condition effect
            m1 = smf.mixedlm(
                'value ~ C(classified_condition)', 
                data=agg, 
                groups=agg['participant_id']
            )
            r1 = m1.fit(method='lbfgs', reml=False, maxiter=200, disp=False)
            
            # Likelihood ratio test
            lrt_statistic = 2.0 * (r1.llf - r0.llf)
            df_diff = k - 1
            p_value = 1.0 - chi2.cdf(lrt_statistic, df_diff)
            
            return {
                'test': 'MixedLM_LRT',
                'statistic': float(lrt_statistic),
                'df': int(df_diff),
                'p_value': float(p_value),
                'significant': bool(p_value < alpha)
            }
        except Exception as e:
            return {'error': f"mixedlm_failed: {type(e).__name__}: {e}"}
    
    def _run_ols_fallback(agg: pd.DataFrame, alpha: float) -> dict:
        """Run OLS with fixed effects and cluster-robust SE"""
        try:
            fit = smf.ols(
                'value ~ C(classified_condition) + C(participant_id)', 
                data=agg
            ).fit(cov_type='cluster', cov_kwds={'groups': agg['participant_id']})
            
            # Extract condition coefficient indices
            param_names = fit.params.index.tolist()
            condition_indices = [
                i for i, name in enumerate(param_names) 
                if name.startswith('C(classified_condition)')
            ]
            
            if not condition_indices:
                return {
                    'test': 'OLS_FE_joint_F',
                    'statistic': np.nan,
                    'df': np.nan,
                    'p_value': np.nan,
                    'significant': False
                }
            
            # Create contrast matrix for joint F-test
            L = np.zeros((len(condition_indices), len(param_names)))
            for row, idx in enumerate(condition_indices):
                L[row, idx] = 1.0
            
            # Joint F-test
            f_test = fit.f_test(L)
            
            return {
                'test': 'OLS_FE_joint_F',
                'statistic': float(f_test.fvalue),
                'df': int(len(condition_indices)),
                'p_value': float(f_test.pvalue),
                'significant': bool(f_test.pvalue < alpha)
            }
        except Exception as e:
            return {'error': f"ols_failed: {type(e).__name__}: {e}"}
    
    # Main analysis loop
    results = []
    
    for metric in metrics:
        aois = _detect_aois_for_metric(df, metric)
        
        if not aois:
            results.append(_create_empty_result(
                metric, None, 'no AOI columns for this metric'
            ))
            continue
        
        for aoi in aois:
            # Prepare data
            agg = _prepare_data(df, aoi, metric, use_log10)
            
            if agg is None:
                results.append(_create_empty_result(
                    metric, aoi, 'no data after dropna'
                ))
                continue
            
            # Check sufficient conditions
            n_conditions = agg['classified_condition'].nunique()
            n_participants = agg['participant_id'].nunique()
            n_observations = len(agg)
            
            if n_conditions < 2:
                results.append(_create_empty_result(
                    metric, aoi, 'fewer than 2 condition levels',
                    n_participants, n_observations
                ))
                continue
            
            # Try MixedLM LRT first
            mlm_result = _run_mixedlm_lrt(agg, n_conditions, alpha)
            
            if 'error' not in mlm_result:
                results.append({
                    'metric': metric,
                    'AOI': aoi,
                    'n_participants': int(n_participants),
                    'n_observations': int(n_observations),
                    **mlm_result
                })
                continue
            
            # Fallback to OLS
            ols_result = _run_ols_fallback(agg, alpha)
            mixed_note = mlm_result.get('error', '')
            
            if 'error' not in ols_result:
                results.append({
                    'metric': metric,
                    'AOI': aoi,
                    'n_participants': int(n_participants),
                    'n_observations': int(n_observations),
                    'note': mixed_note,
                    **ols_result
                })
            else:
                # Both methods failed
                results.append(_create_empty_result(
                    metric, aoi, 
                    f"all_methods_failed: {ols_result['error']} | {mixed_note}",
                    n_participants, n_observations
                ))
    
    return pd.DataFrame(results)

In [ ]:
df = pd.read_csv(str(metrics_csv_path))

results = analyze_aoi_condition_effects(df, metrics=['fixation_count', 'fixation_duration'])

ordered_cols = [
    "metric", "AOI", "test", "statistic", "df",
    "p_value", "significant", "n_participants", "n_observations"
]

if "note" in results.columns:
    ordered_cols.append("note")     

pd.set_option('display.max_rows', 500)     # or None

# Method C: use `display`
from IPython.display import display
display(results)

In [ ]:
# 1) Basic analysis (prefer MixedLM)
res = gzplot.analyze_pairwise_ttests_per_trial(
    input_csv=str(metrics_csv_path),
    metric="fixation_duration",
    outdir=str(cleaned_dir / "fixation_duration_reports"),
    save_figs=False,
    save_tables=False,
    use_log10=False,
    plot_conditions_mode=3,
    plot_use_corrected_p=False,  # flip to False to annotate with raw p-values
)


## ========================================================

### overall MixLM and posthoc

In [ ]:
import importlib
# from gaze_plot_anova_posthoc import create_gaze_anaglysis_all
import gaze_plot_anova_posthoc as gaze_analysis
importlib.reload(gaze_analysis)

In [ ]:
DATA_PATH = str(metrics_csv_path)

fig, ax = gaze_analysis.create_gaze_anaglysis_all(
    DATA_PATH,
    feature=["fixation_count", "fixation_duration"],
    plot_mode="advice",
    # aoi_labels=("Claim Evidence", "Answer", "Reasoning AOI", "Response AOI"),
    # conditions=("no_answer", "correct_reasoning", "incorrect_reasoning"),
    palette="lab",
    show_values=False,
    show_stats=True,  # Whether to show the statistical analysis
    # fdr_method='raw',  # FDR correction method：'fdr_bh' (Benjamini-Hochberg), 'fdr_by' (Benjamini-Yekutieli)
    show_pairwise=True,  # Whether to show pairwise comparisons
    pairwise_p_mode="raw",  # pai
)

# Gaze analysis finish here


## ======================================================================================

## Predictive modeling task
🎯 Goal

Build and evaluate machine learning models to test whether gaze/pupil features can predict human performance.

Output performance metrics, feature importance, and visualization results.

## Model experiment: predict using gaze/pupil features Decision / Confidence / Cognitive Load
Experiment workflow overview:
1. Data preparation: extract the gaze/pupil feature matrix X from `corr_df` / `merged`, with target variables y1=decision_numeric (classification), y2=confidence_rating (regression), y3=cognitive_load (regression). Features are Z-score standardized within CV.\n
2. Models:\n   * Classification: Logistic Regression, RandomForestClassifier\n   * Regression: LinearRegression, XGBoostRegressor (fall back to GradientBoostingRegressor if installation fails)\n
3. 5-fold cross-validation: classification computes Accuracy/AUC/F1; regression computes R²/MAE/RMSE. Results are saved to `prediction_model_metrics.csv`.\n
4. Feature importance: absolute values of linear/Logistic coefficients; feature_importances_ from RF/XGBoost. Combined plots are saved as `feature_importance_[outcome].png`.\n
5. Prediction visualization: classification plots ROC curves (LogReg vs RF); regression plots actual vs predicted scatter plots (two-model comparison).\n
(Missing values are handled automatically via median imputation).

In [ ]:
import sys
from pathlib import Path

# Ensure scripts in this directory can be imported (if the notebook cwd is not `data_analysis_full`)
base = analysis_code_dir
if str(base) not in sys.path:
    sys.path.insert(0, str(base))

import gaze_ml_prediction as gmp


gaze_csv=str(metrics_csv_path)
responses_csv=str(responses_csv_path)
demographic_csv=str(demographic_csv_path)


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Tuple, List, Optional, Union

import pandas as pd

# Compatibility: use rich display in Jupyter, and fall back to `print` in a plain script
try:
    from IPython.display import display  # type: ignore
except Exception:  # pragma: no cover
    def display(x):  # type: ignore
        print(x)


def list_feasible_features(
    gaze_csv: Union[str, Path],
    *,
    sample_rows: Optional[int] = 5000,
    min_nonnull_rate: float = 0.95,
    require_nonzero_std: bool = True,
    sort_by: str = "missing_rate",
    print_top: int = 30,
    suffixes: Optional[Tuple[str, ...]] = None,
    return_report: bool = True,
) -> Union[List[str], Tuple[List[str], pd.DataFrame]]:
    """Output/return all feasible gaze feature columns (usable directly in FEATURES).

    Feasibility criteria:
    - Column name matches AOI_<metricSuffix> (suffix is in suffixes)
    - Usable after numeric conversion (to_numeric coercion)
    - non-null rate >= min_nonnull_rate (default 0.95)
    - Optional: std > 0 (enabled by default, filters constant columns)

    Parameters:
    - sample_rows: by default only reads the first 5000 rows for speed; set to None to read the full file (slower but more accurate)
    - suffixes: if omitted, automatically tries to import METRIC_SUFFIXES from gaze_ml_prediction.py

    Returns:
    - If return_report=True: (features: list[str], report_df: pd.DataFrame)
    - If return_report=False: features: list[str]
    """
    gaze_csv = Path(gaze_csv)
    if not gaze_csv.exists():
        raise FileNotFoundError(str(gaze_csv))

    # 1) Suffix list: prefer reusing `METRIC_SUFFIXES` from the script
    if suffixes is None:
        try:
            from gaze_ml_prediction import METRIC_SUFFIXES as _SUF  # type: ignore
            suffixes = tuple(_SUF)
        except Exception:
            suffixes = (
                "gaze_duration",
                "fixation_duration",
                "fixation_count",
                "saccade_length",
                "saccade_count",
                "pupil_diameter",
                "pupil_diameter_fixation",
                "ttff_ms",
                "ffd_ms",
                "sfd_ms",
            )

    # 2) Read data (sampling for speed)
    if sample_rows is None:
        df = pd.read_csv(gaze_csv, low_memory=False)
    else:
        df = pd.read_csv(gaze_csv, nrows=int(sample_rows), low_memory=False)

    # 3) Find candidate features (matched by suffix)
    cand: List[str] = []
    for c in df.columns:
        cs = str(c)
        for suf in suffixes:
            if cs.endswith("_" + suf):
                cand.append(cs)
                break
    cand = sorted(set(cand))

    if not cand:
        raise ValueError(
            "No candidate gaze feature columns found by suffix. "
            "Please check the CSV column names, or pass suffixes=(...) manually."
        )

    # 4) Compute missingness/variance and filter feasible features
    n = int(df.shape[0])
    rows: List[dict] = []

    for c in cand:
        s = pd.to_numeric(df[c], errors="coerce")
        nn = int(s.notna().sum())
        nonnull_rate = float(nn / n) if n else float("nan")
        missing_rate = float(1.0 - nonnull_rate) if n else float("nan")
        std = float(s.std()) if nn > 1 else float("nan")

        rows.append(
            {
                "feature": c,
                "n_rows": n,
                "n_nonnull": nn,
                "nonnull_rate": nonnull_rate,
                "missing_rate": missing_rate,
                "std": std,
            }
        )

    report = pd.DataFrame(rows)

    if sort_by in report.columns:
        report = report.sort_values(sort_by, ascending=True)
    else:
        report = report.sort_values("missing_rate", ascending=True)

    feasible = report[report["nonnull_rate"] >= float(min_nonnull_rate)].copy()
    if require_nonzero_std:
        feasible = feasible[feasible["std"].fillna(0.0) > 0.0]

    features = feasible["feature"].tolist()

    print(f"Candidate features by suffix: {len(cand)}")
    print(
        f"Feasible features: {len(features)} "
        f"(min_nonnull_rate={min_nonnull_rate}, require_nonzero_std={require_nonzero_std}, sample_rows={sample_rows})"
    )

    if print_top and print_top > 0:
        display(report.head(int(print_top)))

    return (features, report) if return_report else features


# Usage example:
feats, rep = list_feasible_features(gaze_csv, sample_rows=5000, min_nonnull_rate=0.95)
FEATURES = feats
print(feats[:20])


In [ ]:
feats

In [ ]:
SELECTED_FEATURES = [
    "claim_evidence_pupil_diameter",
    "claim_evidence_pupil_diameter_fixation",
    "claim_evidence_saccade_count",
    "claim_evidence_saccade_length",
    "claim_evidence_fixation_duration",
    "claim_evidence_fixation_count",
    "claim_evidence_ttff_ms",
    "answer_aoi_pupil_diameter",
    "answer_aoi_pupil_diameter_fixation",
    "answer_aoi_saccade_count",
    "answer_aoi_saccade_length",
    "answer_aoi_fixation_duration",
    "answer_aoi_fixation_count",
    "answer_aoi_ttff_ms",
    "reasoning_aoi_pupil_diameter",
    "reasoning_aoi_pupil_diameter_fixation",
    "reasoning_aoi_saccade_count",
    "reasoning_aoi_saccade_length",
    "reasoning_aoi_fixation_duration",
    "reasoning_aoi_fixation_count",
    "reasoning_aoi_ttff_ms",
    "response_aoi_pupil_diameter",
    "response_aoi_pupil_diameter_fixation",
    "response_aoi_saccade_count",
    "response_aoi_saccade_length",
    "response_aoi_fixation_duration",
    "response_aoi_fixation_count",
    "response_aoi_ttff_ms",
    # 'classified_condition_numeric', 
    # 'condition_correct_reasoning', 
    # 'condition_incorrect_reasoning', 
    # 'condition_no_answer', 
    # 'condition_no_reasoning'
    ]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
import importlib
import gaze_ml_prediction as gmp
importlib.reload(gmp)

In [ ]:
metrics_df, folds_df, used_features = gmp.run_ml_prediction(
    gaze_csv=gaze_csv,
    responses_csv=responses_csv,
    demographic_csv=demographic_csv,
    cv="loso",
    tuning="grid",
    moe_tuning="none",  # Avoid huge nested tuning for MoE experts (stability).
    preview_only=False,
    targets=["cognitive_load","confidence","accuracy"],  #trust_info, trust_sys, confidence, cognitive_load, accuracy
    features=feats,
    condition=["moe","all","no_answer","correct","incorrect"],  #{correct,incorrect,no_reasoning,all}
    feature_set=["all","demo","gaze"],
    # condition=["correct"],  #{correct,incorrect,no_reasoning,all}
    quantile=0.5,
    threshold_mode="global",   # More stringent: derive the threshold only from the training split
    # models=["logreg","rf","svm"],
    models=["logreg"],  # "gboost","mlp","adaboost","extratrees","gpc"
    save_outputs=True,
    # param_grids={}
    # joblib_backend="threading"
 )

best_df = gmp.make_best_model_table(metrics_df, score_col="f1_mean")
display(best_df)

rank_df = gmp.make_model_ranking_table(metrics_df, score_col="f1_mean")
display(rank_df.head(20))

In [ ]:
metrics_df, folds_df, used_features = gmp.run_ml_prediction(
    gaze_csv=gaze_csv,
    responses_csv=responses_csv,
    demographic_csv=demographic_csv,
    cv="kfold",
    tuning="grid",
    moe_tuning="none",  # Avoid huge nested tuning for MoE experts (stability).
    preview_only=False,
    targets=["cognitive_load","confidence","accuracy"],  #trust_info, trust_sys, confidence, cognitive_load, accuracy
    features=feats,
    condition=["moe","all","no_answer","correct","incorrect"],  #{correct,incorrect,no_reasoning,all}
    feature_set=["all","demo","gaze"],
    # condition=["correct"],  #{correct,incorrect,no_reasoning,all}
    quantile=0.5,
    threshold_mode="global",   # More stringent: derive the threshold only from the training split
    # models=["logreg","rf","svm"],
    models=["logreg"],  # "gboost","mlp","adaboost","extratrees","gpc"
    save_outputs=True,
    # param_grids={}
    # joblib_backend="threading"
 )

best_df = gmp.make_best_model_table(metrics_df, score_col="f1_mean")
display(best_df)

rank_df = gmp.make_model_ranking_table(metrics_df, score_col="f1_mean")
display(rank_df.head(20))

In [ ]:
display(rank_df.head(120))

### feature importance

In [ ]:
import sys
from pathlib import Path
import gaze_ml_prediction as gmp


# Ensure scripts in this directory can be imported (if the notebook cwd is not `data_analysis_full`)
base = analysis_code_dir
if str(base) not in sys.path:
    sys.path.insert(0, str(base))


gaze_csv=str(metrics_csv_path)
responses_csv=str(responses_csv_path)
demographic_csv=str(demographic_csv_path)


SELECTED_FEATURES = [
    "claim_evidence_pupil_diameter",
    "claim_evidence_pupil_diameter_fixation",
    "claim_evidence_saccade_count",
    "claim_evidence_saccade_length",
    "claim_evidence_fixation_duration",
    "claim_evidence_fixation_count",
    "claim_evidence_ttff_ms",
    # "answer_aoi_pupil_diameter",
    # "answer_aoi_pupil_diameter_fixation",
    # "answer_aoi_saccade_count",
    # "answer_aoi_saccade_length",
    # "answer_aoi_fixation_duration",
    # "answer_aoi_fixation_count",
    # "answer_aoi_ttff_ms",
    # "reasoning_aoi_pupil_diameter",
    # "reasoning_aoi_pupil_diameter_fixation",
    # "reasoning_aoi_saccade_count",
    # "reasoning_aoi_saccade_length",
    # "reasoning_aoi_fixation_duration",
    # "reasoning_aoi_fixation_count",
    # "reasoning_aoi_ttff_ms",
    "answer_reasoning_pupil_diameter",
    "answer_reasoning_pupil_diameter_fixation",
    "answer_reasoning_saccade_count",
    "answer_reasoning_saccade_length",
    "answer_reasoning_fixation_duration",
    "answer_reasoning_fixation_count",
    "answer_reasoning_ttff_ms",
    "response_aoi_pupil_diameter",
    "response_aoi_pupil_diameter_fixation",
    "response_aoi_saccade_count",
    "response_aoi_saccade_length",
    "response_aoi_fixation_duration",
    "response_aoi_fixation_count",
    "response_aoi_ttff_ms",
    ]

In [ ]:
import importlib
import gaze_ml_prediction as gmp
importlib.reload(gmp)

In [ ]:
_ = gmp.plot_feature_importance(
    gaze_csv=gaze_csv,
    responses_csv=responses_csv,
    demographic_csv=demographic_csv,
    cv="kfold",
    targets=["cognitive_load","confidence","accuracy"],  #trust_info, trust_sys, confidence, cognitive_load, accuracy
    models=["logreg"],          # Model(s) to inspect
    condition=["all","correct","incorrect"],  #{correct,incorrect,no_reasoning,all}
    feature_set="all",
    features=SELECTED_FEATURES,       # Your `SELECTED_FEATURES`, only for gaze features
    quantile=0.5,
    threshold_mode="global",
    group_features=True,
    group_normalize=True,
    method="shap",                    # auto/native/permutatio
    top_k=10,
    error_bars="ci95",
    save=False,
    show=True,
)

In [ ]:
_ = gmp.plot_feature_importance(
    gaze_csv=gaze_csv,
    responses_csv=responses_csv,
    demographic_csv=demographic_csv,
    cv="loso",
    targets=["cognitive_load","confidence","accuracy"],  #trust_info, trust_sys, confidence, cognitive_load, accuracy
    models=["logreg"],          # Model(s) to inspect
    condition=["all","correct","incorrect"],  #{correct,incorrect,no_reasoning,all}
    feature_set="all",
    features=SELECTED_FEATURES,       # Your `SELECTED_FEATURES`, only for gaze features
    quantile=0.5,
    threshold_mode="global",
    group_features=True,
    group_normalize=True,
    method="shap",                    # auto/native/permutatio
    top_k=10,
    error_bars="ci95",
    save=False,
    show=True,
)

## ==============================================================================================================